# Training and evaluating logistic regression models

In [9]:
import json
from collections import Counter
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, precision_recall_fscore_support
import joblib
from sentence_transformers import SentenceTransformer

data_path = "../data"

with open(f"{data_path}/train_classic_ml_set.json", "r") as f:
    train_classic_ml_set = json.load(f)

with open(f"{data_path}/test_set.json", "r") as f:
    test_set = json.load(f)

BoW_MODEL_PATH = "../models/LR_BoW/3H1L_BoW_LR_model.pkl"
BoW_VECTORIZER_PATH = "../models/LR_BoW/3H1L_BoW_vectorizer.pkl"

SENTENCE_EMBED_MODEL_PATH = "../models/LR_sentence_embeddings/3H1L_sentence_embeddings_LR_model.pkl"
SENTENCE_EMBED_TRAIN_VECTOR_PATH = "../models/LR_sentence_embeddings/3H1L_train_sentence_embeddings.pkl"
SENTENCE_EMBED_TEST_VECTOR_PATH = "../models/LR_sentence_embeddings/3H1L_test_sentence_embeddings.pkl"


### Transform data into appropriate format

Create Python list of dicts

In [ ]:
def flatten_transform(train_or_test_set):

    modified_set = []

    for item in train_or_test_set:
        date = item["date"]
        answer = item["answer"]
        for index, headline in enumerate(item["headlines"]):
            if index == answer:
                fake = 1
            else:
                fake = 0
            entry = {
                "Date": date,
                "Headline": headline,
                "Fake": fake
            }
            modified_set.append(entry)

    return modified_set

train_set_lr_ready = flatten_transform(train_classic_ml_set)
test_set_lr_ready = flatten_transform(test_set)


Transform into dataframe and create X and y classes

In [ ]:
train_df = pd.DataFrame(train_set_lr_ready)
X_train_text = train_df["Headline"]
y_train = train_df["Fake"]

test_df = pd.DataFrame(test_set_lr_ready)
X_test_text = test_df["Headline"]
y_test = test_df["Fake"] # May not actually use this given quiz level accuracy is primary metric

### BoW TF-IDF LR

Model training or loading

In [ ]:
try:
    clf_BoW = joblib.load(BoW_MODEL_PATH)
    vectorizer = joblib.load(BoW_VECTORIZER_PATH)
    print("Loaded model and vectorizer from disk.")

except FileNotFoundError:

    print("No saved model found, training a new one...")

    # Create TF-IDF vectorizer
    vectorizer = TfidfVectorizer(
        max_features=5000,  # cap number of features for speed
        ngram_range=(1, 2)  # include single words and short phrases
    )

    # Fit on training data, transform both train and test sets
    X_train_BoW = vectorizer.fit_transform(X_train_text)
    X_test_BoW = vectorizer.transform(X_test_text)

    # Initialise model
    clf_BoW = LogisticRegression(max_iter=1000, random_state=42)

    # Train model
    clf_BoW.fit(X_train_BoW, y_train)

    joblib.dump(clf_BoW, BoW_MODEL_PATH)
    joblib.dump(vectorizer, BoW_VECTORIZER_PATH)
    print("Model and vectorizer saved.")

else:
    X_train_BoW = vectorizer.transform(X_train_text)
    X_test_BoW = vectorizer.transform(X_test_text)

print("TF-IDF matrix shape (train):", X_train_BoW.shape)
print("TF-IDF matrix shape (test):", X_test_BoW.shape)

No saved model found, training a new one...
Model and vectorizer saved.
TF-IDF matrix shape (train): (1080, 5000)
TF-IDF matrix shape (test): (268, 5000)


Evaluate

In [ ]:
y_pred_BoW = clf.predict(X_test_BoW)
y_fake_prob_BoW = clf.predict_proba(X_test_BoW)[:, 1]

In [ ]:
test_df["Fake Prob BoW"] = y_fake_prob_BoW
test_by_quiz_df = test_df.groupby("Date")

correct = 0

for date, quiz_df in test_by_quiz_df:
    predicted_index = quiz_df["Fake Prob BoW"].idxmax()
    actual_index = quiz_df["Fake"].idxmax()

    if predicted_index == actual_index:
        correct += 1

accuracy_BoW = correct / len(test_set)

print(f"Quiz-level accuracy: {accuracy_BoW:.2%}")

Quiz-level accuracy: 43.28%


### Sentence Embeddings LR

In [13]:
try:
    clf_SE = joblib.load(SENTENCE_EMBED_MODEL_PATH)
    X_train_SE = joblib.load(SENTENCE_EMBED_TRAIN_VECTOR_PATH)
    X_test_SE = joblib.load(SENTENCE_EMBED_TEST_VECTOR_PATH)
    print("Loaded model and embeddings from disk.")

except FileNotFoundError:

    print("No saved model found, training a new one...")

    # Create embedding model
    model = SentenceTransformer("all-MiniLM-L6-v2")

    # Embed both train and test sets
    X_train_SE = model.encode(list(X_train_text))
    X_test_SE = model.encode(list(X_test_text))

    # Initialise LR model
    clf_SE = LogisticRegression(max_iter=1000, random_state=42)

    # Train LR model
    clf_SE.fit(X_train_SE, y_train)

    joblib.dump(clf_SE, SENTENCE_EMBED_MODEL_PATH)
    joblib.dump(X_train_SE, SENTENCE_EMBED_TRAIN_VECTOR_PATH)
    joblib.dump(X_test_SE, SENTENCE_EMBED_TEST_VECTOR_PATH)
    print("Model and embeddings saved.")

print("Embeddings matrix shape (train):", X_train_SE.shape)
print("Embeddings matrix shape (test):", X_test_SE.shape)

Loaded model and embeddings from disk.
Embeddings matrix shape (train): (1080, 384)
Embeddings matrix shape (test): (268, 384)


In [14]:
y_pred_SE = clf_SE.predict(X_test_SE)
y_fake_prob_SE = clf_SE.predict_proba(X_test_SE)[:, 1]

In [17]:
test_df["Fake Prob SE"] = y_fake_prob_SE
test_by_quiz_df = test_df.groupby("Date")

correct = 0

for date, quiz_df in test_by_quiz_df:
    predicted_index = quiz_df["Fake Prob SE"].idxmax()
    actual_index = quiz_df["Fake"].idxmax()

    if predicted_index == actual_index:
        correct += 1

accuracy_SE = correct / len(test_set)

print(f"Quiz-level accuracy: {accuracy_SE:.2%}")

Quiz-level accuracy: 46.27%
